In [4]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import os
import zipfile

In [5]:
data_folder = None
zip_file = None
required_classes = {'EarlyBloom', 'Healthy', 'MatureBud', 'Wilted', 'YoungBud'}

for root, dirs, files in os.walk('/kaggle/input'):
    if required_classes.issubset(set(dirs)):
        data_folder = root
        break
    for file_name in files:
        if file_name.lower().endswith('.zip') and 'sunflower' in file_name.lower():
            zip_file = os.path.join(root, file_name)

if data_folder is None and zip_file is not None:
    extract_folder = '/kaggle/working/sunflower_dataset'
    os.makedirs(extract_folder, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as zip_data:
        zip_data.extractall(extract_folder)

    for root, dirs, files in os.walk(extract_folder):
        if required_classes.issubset(set(dirs)):
            data_folder = root
            break

if data_folder is None:
    raise FileNotFoundError('connect the Sunflower Dataset by adding input.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Dataset folder:', data_folder)

Device: cpu
Dataset folder: /kaggle/input/datasets/anikatahsinadiba/anikacvpr/Sunflower Compressed


In [6]:
class_names = sorted([
    folder for folder in os.listdir(data_folder)
    if os.path.isdir(os.path.join(data_folder, folder))
])

print('Classes:', class_names)
print('Number of classes:', len(class_names))
assert len(class_names) == 5, 'Sunflower Dataset = 5'

tr_images, tr_targets = [], []
val_images, val_targets = [], []

np.random.seed(42)

for label, class_name in enumerate(class_names):
    class_folder = os.path.join(data_folder, class_name)
    image_files = sorted([
        os.path.join(class_folder, file_name)
        for file_name in os.listdir(class_folder)
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    np.random.shuffle(image_files)
    split_index = int(len(image_files) * 0.8)

    for i, image_path in enumerate(image_files):
        with Image.open(image_path) as image:
            image = np.array(image.convert('RGB').resize((32, 32)))

        if i < split_index:
            tr_images.append(image)
            tr_targets.append(label)
        else:
            val_images.append(image)
            val_targets.append(label)

tr_images = np.array(tr_images, dtype=np.uint8)
tr_targets = np.array(tr_targets, dtype=np.int64)
val_images = np.array(val_images, dtype=np.uint8)
val_targets = np.array(val_targets, dtype=np.int64)

print('Training image shape:', tr_images.shape)
print('Validation image shape:', val_images.shape)
print('Training labels:', tr_targets.shape)
print('Validation labels:', val_targets.shape)

Classes: ['EarlyBloom', 'Healthy', 'MatureBud', 'Wilted', 'YoungBud']
Number of classes: 5
Training image shape: (4087, 32, 32, 3)
Validation image shape: (1024, 32, 32, 3)
Training labels: (4087,)
Validation labels: (1024,)


In [7]:
batch_size_no = 32
learning_rate = 1e-3
epoch_number = 30

In [8]:
class SunflowerDataset(Dataset):
    def __init__(self, x, y):
        x = torch.tensor(x)     # N, H, W, C
        x = x.float() / 255     # Normalization: 0..255 -> 0..1
        x = x.permute(0, 3, 1, 2)  # N, C, H, W
        x = x.reshape(-1, 3, 32, 32)

        y = torch.tensor(y)
        self.x, self.y = x, y

    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)

    def __len__(self):
        return len(self.x)


def get_data():
    train = SunflowerDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=batch_size_no, shuffle=True)
    val = SunflowerDataset(val_images, val_targets)
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=True)
    return trn_dl, val_dl

In [9]:
trn_dl, val_dl = get_data()
print('Number of training batches:', len(trn_dl))
print('Number of validation batches:', len(val_dl))

Number of training batches: 128
Number of validation batches: 1


In [10]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.activation import ReLU
from torch.optim import SGD, Adam

def get_model():
    model = nn.Sequential(
        # Feature Extractor / Convolution layers
        # 1st conv
        nn.Conv2d(3, 40, 3, padding=1), # 32x32
        nn.ReLU(),                       # 32x32
        nn.MaxPool2d(2),                 # 16x16
        # 2nd conv
        nn.Conv2d(40, 65, 3, padding=1), # 16x16
        nn.ReLU(),                       # 16x16
        nn.MaxPool2d(2),                 # 8x8
        # 3rd conv
        nn.Conv2d(65, 70, 3, padding=1), # 8x8
        nn.ReLU(),                       # 8x8
        nn.MaxPool2d(2),                 # 4x4
        # Classifier / Neural Network
        nn.Flatten(),
        nn.Linear(70 * 4 * 4, 256),
        nn.ReLU(),
        nn.Linear(256, 5)
    ).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=learning_rate)
    return model, loss_fn, optimizer

In [11]:
def train_one_epoch(data_loader, model, loss_fn, optimizer):
    model.train()
    epoch_losses, epoch_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        prediction = model(x)
        batch_loss = loss_fn(prediction, y)
        batch_loss.backward()
        optimizer.step()

        epoch_losses.append(batch_loss.item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        epoch_accuracies.extend(is_correct)
    return np.mean(epoch_losses), np.mean(epoch_accuracies)

In [12]:
@torch.no_grad()
def validation(data_loader, model, loss_fn):
    model.eval()
    val_losses, val_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        prediction = model(x)
        val_losses.append(loss_fn(prediction, y).item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        val_accuracies.extend(is_correct)
    return np.mean(val_losses), np.mean(val_accuracies)

In [13]:
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Trainable parameters: 353866


In [14]:
from torch import optim
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    factor=0.5,
    patience=0,
    threshold=0.001,
    min_lr=1e-5,
    threshold_mode='abs'
)

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(epoch_number):
    epoch_loss, epoch_accuracy = train_one_epoch(trn_dl, model, loss_fn, optimizer)
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    v_loss, v_acc = validation(val_dl, model, loss_fn)
    val_losses.append(v_loss)
    val_accuracies.append(v_acc)

    scheduler.step(v_loss)

    print(f'Epoch [{epoch + 1}/{epoch_number}]'
          f' | Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_accuracy*100:.2f}%'
          f' | Val Loss: {v_loss:.4f} | Val Accuracy: {v_acc*100:.2f}%')

Epoch [1/30] | Train Loss: 1.0469 | Train Accuracy: 53.36% | Val Loss: 0.6911 | Val Accuracy: 71.29%
Epoch [2/30] | Train Loss: 0.6507 | Train Accuracy: 72.69% | Val Loss: 0.5669 | Val Accuracy: 77.44%
Epoch [3/30] | Train Loss: 0.4944 | Train Accuracy: 79.99% | Val Loss: 0.5791 | Val Accuracy: 74.80%
Epoch [4/30] | Train Loss: 0.3734 | Train Accuracy: 85.64% | Val Loss: 0.3774 | Val Accuracy: 85.16%
Epoch [5/30] | Train Loss: 0.3204 | Train Accuracy: 87.35% | Val Loss: 0.3690 | Val Accuracy: 86.13%
Epoch [6/30] | Train Loss: 0.3029 | Train Accuracy: 88.43% | Val Loss: 0.3356 | Val Accuracy: 87.50%
Epoch [7/30] | Train Loss: 0.2683 | Train Accuracy: 89.85% | Val Loss: 0.3340 | Val Accuracy: 88.28%
Epoch [8/30] | Train Loss: 0.2480 | Train Accuracy: 90.85% | Val Loss: 0.3105 | Val Accuracy: 87.11%
Epoch [9/30] | Train Loss: 0.2412 | Train Accuracy: 90.82% | Val Loss: 0.3121 | Val Accuracy: 87.99%
Epoch [10/30] | Train Loss: 0.1872 | Train Accuracy: 93.30% | Val Loss: 0.3273 | Val Accura